In [9]:
import requests_cache
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [10]:
session = requests_cache.CachedSession(
    cache_name='Extra', use_cache_dir=True, expire_after=604800)

# south_america = {
#     'USAF': 'https://fbref.com/en/comps/182/2023/2023-NWSL-Stats',
#     'Ecuadorm': 'https://fbref.com/en/comps/58/2023/2023-Serie-A-Stats',
#     'Perum': 'https://fbref.com/en/comps/44/2023/2023-Liga-1-Stats',
#     'Uruguaym': 'https://fbref.com/en/comps/45/2023/2023-Primera-Division-Stats',
#     'USAm': 'https://fbref.com/en/comps/22/2023/2023-Major-League-Soccer-Stats',
# }

In [11]:
def get_team_links(b_data, L_soup):
    standings_table = L_soup.select('table.stats_table')[0]
    td_tags = standings_table.find_all('td', {"data-stat": 'team'})

    all_team_names = [a.get_text(strip=True)
                      for td in td_tags for a in td.find_all('a')]

    links = standings_table.find_all('a')
    links = [l.get("href") for l in links]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]

    team_ids = [re.search(r'/squads/([^/]+)/', t_url).group(1)
                for t_url in team_urls if re.search(r'/squads/([^/]+)/', t_url)]
    team_name_links = [t_url.split("/")[-1].replace("-Stats", "")
                       for t_url in team_urls]

    return team_urls, all_team_names, team_ids, team_name_links

In [12]:
def get_team_data(data, counter, team_name, fteam, opponent):

    if counter == 0:
        ScoresFixtures = pd.read_html(data.text, match="Scores & Fixtures")[0]
        ScoresFixtures.drop(['Match Report', 'Notes'], axis=1, inplace=True)
        fteam.append(ScoresFixtures)

    elif counter == 1:
        try:
            Shooting = pd.read_html(data.text, match="Shooting", header=1)
            # Shooting[0] = Shooting[0].rename(columns=lambda x: f'{team_name}_' + x)
            Shooting[1] = Shooting[1].rename(columns=lambda x: f'O_' + x)
            fteam.append(Shooting[0].iloc[:-1, 11:-1])
            opponent.append(Shooting[1].iloc[:-1, 11:-1])

        except Exception as e:
            print('The problem is ', e)
    elif counter == 2:
        try:
            Goalkeeping = pd.read_html(
                data.text, match="Goalkeeping", header=1)
            # Goalkeeping[0] = Goalkeeping[0].rename(columns=lambda x: f'{team_name}_' + x)
            Goalkeeping[1] = Goalkeeping[1].rename(columns=lambda x: f'O_' + x)

            Goalkeeping[0].drop('GA.1', axis=1, inplace=True)
            Goalkeeping[1].drop('O_GA.1', axis=1, inplace=True)

            fteam.append(Goalkeeping[0].iloc[:-1, 10:-1])
            opponent.append(Goalkeeping[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    elif counter == 3:
        try:
            Passing = pd.read_html(data.text, match="Passing", header=1)
            # Passing[0] = Passing[0].rename(columns=lambda x: f'{team_name}_' + x)
            Passing[1] = Passing[1].rename(columns=lambda x: f'O_' + x)

            fteam.append(Passing[0].iloc[:-1, 10:-1])
            opponent.append(Passing[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    elif counter == 4:
        try:
            PassTypes = pd.read_html(data.text, match="Pass Types", header=1)
            # PassTypes[0] = PassTypes[0].rename(columns=lambda x: f'{team_name}_' + x)
            PassTypes[1] = PassTypes[1].rename(columns=lambda x: f'O_' + x)

            fteam.append(PassTypes[0].iloc[:-1, 10:-1])
            opponent.append(PassTypes[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 5:
        try:
            GoalShotCreation = pd.read_html(
                data.text, match="Goal and Shot Creation", header=1)
            # GoalShotCreation[0] = GoalShotCreation[0].rename(columns=lambda x: f'{team_name}_' + x)
            GoalShotCreation[1] = GoalShotCreation[1].rename(
                columns=lambda x: f'O_' + x)

            fteam.append(GoalShotCreation[0].iloc[:-1, 10:-1])
            opponent.append(GoalShotCreation[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 6:
        try:
            DefensiveActions = pd.read_html(
                data.text, match="Defensive Actions", header=1)
            # DefensiveActions[0] = DefensiveActions[0].rename(columns=lambda x: f'{team_name}_' + x)
            DefensiveActions[1] = DefensiveActions[1].rename(
                columns=lambda x: f'O_' + x)

            fteam.append(DefensiveActions[0].iloc[:-1, 10:-1])
            opponent.append(DefensiveActions[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 7:
        try:
            Possession = pd.read_html(data.text, match="Possession", header=1)
            # Possession[0] = Possession[0].rename(columns=lambda x: f'{team_name}_' + x)
            Possession[1] = Possession[1].rename(columns=lambda x: f'O_' + x)

            fteam.append(Possession[0].iloc[:-1, 10:-1])
            opponent.append(Possession[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)

    elif counter == 8:
        try:
            MiscellaneousStats = pd.read_html(
                data.text, match="Miscellaneous Stats", header=1)
            # MiscellaneousStats[0] = MiscellaneousStats[0].rename(columns=lambda x: f'{team_name}_' + x)
            MiscellaneousStats[1] = MiscellaneousStats[1].rename(
                columns=lambda x: f'O_' + x)

            fteam.append(MiscellaneousStats[0].iloc[:-1, 10:-1])
            opponent.append(MiscellaneousStats[1].iloc[:-1, 10:-1])

        except Exception as e:
            print('The problem is ', e)
    return fteam, opponent

In [5]:
# session.cache.clear()

In [13]:
def league_proccess(league_link):
    base_data = session.get(league_link)
    League_soup = BeautifulSoup(base_data.text, features="lxml")

    seasons_name = [
        # "2023-2024",
        # "2022-2023",
        # "2021-2022",
        "2020-2021",
        "2019-2020",
        "2018-2019",
        "2017-2018"
                    ]
    # seasons_name = [
    #     "2021",
    #     "2022",
    #     "2023"]
    team_urls, all_team_names, team_id, team_name_l = get_team_links(
        base_data, League_soup)

    return seasons_name, team_urls, all_team_names, team_id, team_name_l

In [14]:
def team_stats(team_link, all_team_names, index, team_id, seasons_name, season_index, team_name_l):
    print(team_name_l[index])
    data = session.get(team_link)
    print(data.from_cache)
    if data.from_cache == False:
        time.sleep(60)
        
    soup = BeautifulSoup(data.text, features="lxml")
    mdiv = soup.select('div', {'id': 'content'})[0]
    link = mdiv.find_all('a')
    link = [l.get("href") for l in link]
    link = [f"https://fbref.com" + l for l in link if l != None]
    
    
    
    # tables_urls = get_tables_links(data, soup)
    tables_urls = [f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/schedule/{team_name_l[index]}-Scores-and-Fixtures-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/shooting/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/keeper/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/passing/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/passing_types/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/gca/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/defense/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/possession/{team_name_l[index]}-Match-Logs-All-Competitions',
                    f'https://fbref.com/en/squads/{team_id[index]}/{seasons_name[season_index]}/matchlogs/all_comps/misc/{team_name_l[index]}-Match-Logs-All-Competitions']

    # tables_urls = [f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/schedule/{team_name_l[index]}-Scores-and-Fixtures-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/shooting/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/keeper/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/passing/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/passing_types/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/gca/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/defense/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/possession/{
    #                    team_name_l[index]}-Match-Logs-All-Competitions',
    #                f'https://fbref.com/en/squads/{team_id[index]}/2023-2024/matchlogs/all_comps/misc/{team_name_l[index]}-Match-Logs-All-Competitions']



    if all(element in link for element in tables_urls[1:]):
        print('True. All links are in this page.')
        TeamStatistics = pd.DataFrame()
        counter = 0

        fteam = []
        opponent = []
        for url in tables_urls:
            res = session.get(url)
            fteam, opponent = get_team_data(
                res, counter, all_team_names[index], fteam, opponent)
            counter += 1
            time.sleep(20)

        fteamdf = pd.concat(fteam, axis=1)
        opponentdf = pd.concat(opponent, axis=1)
        TeamStatistics = pd.concat([fteamdf, opponentdf], axis=1)
        TeamStatistics.insert(9, "Team", all_team_names[index])
        return TeamStatistics
    else:
        return pd.DataFrame()

⛔ Get all data except the America continent

In [15]:
seasons = [
    # "2023-2024/2023-2024-",
    # "2022-2023/2022-2023-",
    # "2021-2022/2021-2022-",
    "2020-2021/2020-2021-",
    "2019-2020/2019-2020-",
    "2018-2019/2018-2019-",
    "2017-2018/2017-2018-"]

for season_index, season in enumerate(seasons):

    league_urls = {
    # 'Laligaf': f"https://fbref.com/en/comps/230/{season}Liga-F-Stats",
    # 'Bundesligaf': f"https://fbref.com/en/comps/183/{season}Frauen-Bundesliga-Stats",
    # 'SerieAf': f"https://fbref.com/en/comps/208/{season}Serie-A-Stats",
    # 'League1f': f"https://fbref.com/en/comps/193/{season}Division-1-Feminine-Stats",
    # 'Australiaf': f'https://fbref.com/en/comps/196/{season}A-League-Women-Stats',
    # 'Austriam': f'https://fbref.com/en/comps/56/{season}Austrian-Bundesliga-Stats',
    # 'Czechm': f'https://fbref.com/en/comps/66/{season}Czech-First-League-Stats', just have 23-24
    # 'Bulgariam': f'https://fbref.com/en/comps/67/{season}Bulgarian-First-League-Stats',
    # 'Greecem': f'https://fbref.com/en/comps/27/{season}Super-League-Greece-Stats',
    # 'Mexicom': f'https://fbref.com/en/comps/31/{season}Liga-MX-Stats',
    # 'Scotlandm': f'https://fbref.com/en/comps/40/{season}Scottish-Premiership-Stats',
    # 'LaligaBm': f'https://fbref.com/en/comps/12/{season}La-Liga-Stats',
    # 'SerieBm': f'https://fbref.com/en/comps/18/{season}Serie-B-Stats',
    'Bundesliga2m': f'https://fbref.com/en/comps/33/{season}2-Bundesliga-Stats',
    'League2m': f'https://fbref.com/en/comps/60/{season}Ligue-2-Stats'
    }
    print(season)
    for league_name, league_link in league_urls.items():
        
        fc = ["2020-2021/2020-2021-", "2019-2020/2019-2020-", "2018-2019/2018-2019-", "2017-2018/2017-2018-"]
        # sc = ["2022-2023/2022-2023-","2021-2022/2021-2022-","2020-2021/2020-2021-","2019-2020/2019-2020-","2018-2019/2018-2019-","2017-2018/2017-2018-"]
        tc = ["2019-2020/2019-2020-","2018-2019/2018-2019-","2017-2018/2017-2018-"]
        fd = ["2021-2022/2021-2022-", "2020-2021/2020-2021-", "2019-2020/2019-2020-", "2018-2019/2018-2019-", "2017-2018/2017-2018-"]
        
        if league_name == 'Laligaf' and season in fd:
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        elif league_name == 'Bundesligaf' and season in fc:
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        elif league_name == 'SerieAf' and season in tc:
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        elif league_name == 'League1f' and season in fc:
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        elif league_name == 'Australiaf' and season == "2017-2018/2017-2018-":
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        # elif league_name == 'Czechm' and season in [sc]:
        #     continue
        elif league_name == 'Mexicom' and season == "2017-2018/2017-2018-":
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        elif league_name == 'Scotlandm' and season == "2017-2018/2017-2018-":
            print(f"The {league_name} doesn't have the season {season}. ")
            continue
        elif league_name == 'SerieBm' and season in fd:
            print(f"The {league_name} doesn't have the season {season}. ")
            continue

        each_league = []
        seasons_name, team_urls, all_team_names, team_id, team_name_l = league_proccess(
            league_link)
        
        hmt = len(team_urls)
        for index, team_link in enumerate(team_urls):
            TeamStatistics = team_stats(
                team_link, all_team_names, index, team_id, seasons_name, season_index, team_name_l)
            if not TeamStatistics.empty:
                each_league.append(TeamStatistics)
                print(TeamStatistics.shape)
                time.sleep(30)
            else:
                print('The dataframe is empty.')
                time.sleep(60)
            print(f'** {hmt} ** Team remain in the league {league_name} and season {seasons_name[season_index]}.')
            hmt -= 1
            print('*' * 100)
        leagues = pd.concat(each_league, axis=0)
        leagues.to_csv(f"ExtraData/{league_name}_{seasons_name[season_index]}.csv", mode='w', index=False, encoding="utf-8")

2020-2021/2020-2021-
Red-Bull-Salzburg
True
True. All links are in this page.
(42, 308)
** 12 ** Team remain in the league Austriam and season 2020-2021.
****************************************************************************************************
Rapid-Wien
True
True. All links are in this page.
(40, 308)
** 11 ** Team remain in the league Austriam and season 2020-2021.
****************************************************************************************************
Sturm-Graz
True
The dataframe is empty.
** 10 ** Team remain in the league Austriam and season 2020-2021.
****************************************************************************************************
Hartberg
True
The dataframe is empty.
** 9 ** Team remain in the league Austriam and season 2020-2021.
****************************************************************************************************
LASK
True
True. All links are in this page.
(40, 308)
** 8 ** Team remain in the league Austriam and season

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
data_1 = pd.read_csv('ExtraData/Australiaf_2021-2022.csv')
data_2 = pd.read_csv('ExtraData/Austriam_2021-2022.csv')
data_3 = pd.read_csv('ExtraData/Bulgariam_2021-2022.csv')
data_4 = pd.read_csv('ExtraData/Bundesliga2m_2021-2022.csv')
data_5 = pd.read_csv('ExtraData/Bundesligaf_2021-2022.csv')
data_6 = pd.read_csv('ExtraData/Greecem_2021-2022.csv')
data_7 = pd.read_csv('ExtraData/LaligaBm_2021-2022.csv')
# data_8 = pd.read_csv('ExtraData/Laligaf_2021-2022.csv')
data_9 = pd.read_csv('ExtraData/League1f_2021-2022.csv')
data_10 = pd.read_csv('ExtraData/Mexicom_2021-2022.csv')
data_11 = pd.read_csv('ExtraData/Scotlandm_2021-2022.csv')
data_12 = pd.read_csv('ExtraData/SerieAf_2021-2022.csv')
# data_13 = pd.read_csv('ExtraData/SerieBm_2021-2022.csv')
data_14 = pd.read_csv('ExtraData/League2m_2021-2022.csv')


all_dfs = [data_1, data_2, data_3, data_4, data_5, data_6, data_7,
        #    data_8,
           data_9, data_10,
           data_11, data_12,
        #    data_13,
           data_14]


complete_dataset = pd.concat(all_dfs, axis=0)
complete_dataset.to_csv('ExtraData/2027-2022.csv',
                        mode='w', index=False, encoding="utf-8")

⛔ The south American Leagues that have different types of urls

In [ ]:
seasons = [
    # "2021/2021",
    # "2022/2022",
    "2023/2023"]
for season_index, season in enumerate(seasons):
    dds = []

    league_urls = {
        'PrimeraDivision': f'https://fbref.com/en/comps/21/{season}-Primera-Division-Stats',
        'SerieABrazil': f'https://fbref.com/en/comps/24/{season}-Serie-A-Stats',
    }
    print(season)
    for league_name, league_link in league_urls.items():
        e_league = []
        seasons_name, team_urls, all_team_names, team_id, team_name_l = league_proccess(
            league_link)

        for index, team_link in enumerate(team_urls):
            TeamStatistics = team_stats(
                team_link, all_team_names, index, team_id, seasons_name, season_index, team_name_l)
            print(TeamStatistics.shape)
            dds.append(TeamStatistics)
            e_league.append(TeamStatistics)
            print('*' * 100)
            time.sleep(200)
        eax_league = pd.concat(e_league, axis=0)
        if league_name == 'PrimeraDivision':
            eax_league.to_csv(f"Datasets/PrimeraDivision2023.csv",
                              mode='w', index=False, encoding="utf-8")
        elif league_name == 'SerieABrazil':
            eax_league.to_csv(f"Datasets/SerieABrazil2023.csv",
                              mode='w', index=False, encoding="utf-8")

    #  f_dds = pd.concat(dds, axis=0)
    #  TeamStatistics.to_csv(f"Datasets/_{seasons[season_index]}_.csv", mode='w', index=False, encoding="utf-8")